# 02c · EverydayScents: R2 handcrafted features

**Purpose.** Compute k = 12 features per sensor (R2). Fit the 4 heads on both tasks with the same folds and seeds as 01b.

**Inputs**
- EverydayScents `X_signals.npy` (144, 11, 210) + `meta.csv`

**Outputs**
- F_hand: (144, 11, 12) → (144, 132)
- `results/02c_es_handcrafted.csv`

In [1]:
import sys
sys.path.append("../..")          # project root, so that `import src` works

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path("../..")
from src.cv import TASKS, load_baseline_level, load_processed, precomputed, run_cv, save_results, summarize
from src.features.handcrafted import FEATURE_NAMES, handcrafted_features
DATASET = "es"
REP = "R2_handcrafted"
NAME = "02c_es_handcrafted"          # results/02c_es_handcrafted.csv

X, meta, info = load_processed(DATASET)    # X: (N, C, T) baseline corrected
print(X.shape, meta["y_multi"].value_counts().sort_index().tolist())

(144, 11, 210) [33, 39, 50, 22]


## R2: 12 handcrafted features per sensor
Nothing is fitted, so the features are computed once for all samples.

In [2]:
levels = load_baseline_level(DATASET)     # (N, C) resting level before baseline correction
F = handcrafted_features(X, levels, info["n_base"], info["exposure_end"])   # (N, C * 12) = (144, 132)
assert F.shape == (144, 132) and np.isfinite(F).all()
pd.DataFrame(F[:, :12], columns=FEATURE_NAMES).describe().T   # the 12 features of the first sensor

,count,mean,std,min,25%,50%,75%,max
baseline_level,144.0,0.174249,0.031234,1.223447e-01,0.153631,0.175979,1.852460e-01,0.349278
steady_state,144.0,0.250492,0.385688,-7.259844e-03,0.002076,0.065928,3.830578e-01,1.807211
normalized,144.0,1.597615,2.581825,-3.322542e-02,0.012113,0.370926,1.636627e+00,10.310960
peak,144.0,0.278105,0.411842,-2.136318e-02,0.004597,0.079246,4.946783e-01,1.813863
time_to_peak,144.0,76.465278,49.908015,0.000000e+00,33.000000,81.500000,1.242500e+02,149.000000
rise_slope,144.0,0.003682,0.007877,-1.521926e-03,0.000023,0.000485,2.475341e-03,0.048388
decay_slope,144.0,-0.000510,0.001349,-1.008566e-02,-0.000473,-0.000076,6.871125e-20,0.000400
auc,144.0,34.039369,53.345143,-9.118051e-01,0.254317,8.069689,3.502108e+01,215.854002
ema_max_0.1,144.0,0.011310,0.017574,-3.042016e-11,0.000264,0.002061,1.327992e-02,0.063496
ema_min_0.1,144.0,-0.001500,0.002784,-2.080631e-02,-0.001594,-0.000464,-1.860164e-04,0.000038


## Cross validation: 4 heads × 5 folds × 3 seeds, both tasks
Everything that is fitted (scaler, heads, hyperparameters) sees the training fold only.

In [3]:
scores, preds = [], []
for task in TASKS:                                   # "multi", then "binary"
    s, p = run_cv(F, meta, DATASET, task, REP, precomputed)
    scores.append(s)
    preds.append(p)
scores, preds = pd.concat(scores, ignore_index=True), pd.concat(preds, ignore_index=True)
save_results(scores, preds, NAME)
print("saved results/" + NAME + ".csv")

es multi R2_handcrafted seed 0 fold 0: logreg 0.627  svm 0.602  rf 1.000  mlp 0.627
es multi R2_handcrafted seed 0 fold 1: logreg 0.796  svm 0.823  rf 0.831  mlp 0.759
es multi R2_handcrafted seed 0 fold 2: logreg 0.705  svm 0.769  rf 0.914  mlp 0.688
es multi R2_handcrafted seed 0 fold 3: logreg 0.657  svm 0.585  rf 0.717  mlp 0.585
es multi R2_handcrafted seed 0 fold 4: logreg 0.750  svm 0.521  rf 0.900  mlp 0.491
es multi R2_handcrafted seed 1 fold 0: logreg 0.759  svm 0.548  rf 0.971  mlp 0.564
es multi R2_handcrafted seed 1 fold 1: logreg 0.809  svm 0.578  rf 0.837  mlp 0.691
es multi R2_handcrafted seed 1 fold 2: logreg 0.620  svm 0.581  rf 0.942  mlp 0.620
es multi R2_handcrafted seed 1 fold 3: logreg 0.576  svm 0.543  rf 0.729  mlp 0.538
es multi R2_handcrafted seed 1 fold 4: logreg 0.663  svm 0.579  rf 0.887  mlp 0.730
es multi R2_handcrafted seed 2 fold 0: logreg 0.738  svm 0.784  rf 0.963  mlp 0.831
es multi R2_handcrafted seed 2 fold 1: logreg 0.578  svm 0.535  rf 0.844  ml

## Results: mean ± std over 3 seeds × 5 folds

In [4]:
print("macro F1"); display(summarize(scores))
print("balanced accuracy"); display(summarize(scores, "balanced_acc"))

macro F1


head                                  logreg            mlp             rf  \
dataset task   rep                                                           
es      binary R2_handcrafted  0.905 ± 0.034  0.934 ± 0.044  0.962 ± 0.055   
        multi  R2_handcrafted  0.679 ± 0.078  0.628 ± 0.099  0.871 ± 0.089   

head                                     svm  
dataset task   rep                            
es      binary R2_handcrafted  0.884 ± 0.111  
        multi  R2_handcrafted  0.616 ± 0.095

balanced accuracy


head                                  logreg            mlp             rf  \
dataset task   rep                                                           
es      binary R2_handcrafted  0.904 ± 0.032  0.935 ± 0.041  0.961 ± 0.055   
        multi  R2_handcrafted  0.696 ± 0.068  0.655 ± 0.083  0.880 ± 0.079   

head                                     svm  
dataset task   rep                            
es      binary R2_handcrafted  0.878 ± 0.108  
        multi  R2_handcrafted  0.655 ± 0.079

In [5]:
print("ROC AUC, binary task"); summarize(scores[scores["task"] == "binary"], "roc_auc")

ROC AUC, binary task


,,head,logreg,mlp,rf,svm
dataset,task,rep,,,,
es,binary,R2_handcrafted,0.949 ± 0.042,0.955 ± 0.052,0.978 ± 0.036,0.976 ± 0.027
